#### **Business rate revaluations 2026 by LAD**

This file maps changes in rateable values between 2023 and 2026 across Local Authorities.

Data comes from the Valuation Office Agency
https://www.gov.uk/government/statistics/non-domestic-rating-change-in-rateable-value-of-rating-lists-england-and-wales-2026-revaluation-draft-list

In [5]:
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
import numpy as np
import requests
import zipfile
import os
from pathlib import Path

def load_voa_data(excel_path):
    """Load and clean VOA spreadsheet data from sheet 6"""
    try:
        # Read from sheet 6 (0-indexed, so sheet_name=5)
        df = pd.read_excel(excel_path, sheet_name=5, header=None)
        print(f"Loaded data from sheet 6 with shape: {df.shape}")
        
        # Define column names based on your data structure
        column_names = [
            'Area_Type', 'Col2', 'LA_Code_Old', 'ONS_Code', 'Area_Name',
            'Properties_2023', 'RV_2023', 'RV_Mean_2023', 'RV_Median_2023',
            'RV_2026', 'RV_Mean_2026', 'RV_Median_2026', 
            'Change_Absolute', 'Change_Percent'
        ]
        
        df.columns = column_names
        
        # Filter for Local Authority areas only (LAUA)
        df_filtered = df[df['Area_Type'] == 'LAUA'].copy()
        
        # Clean area names - remove anything after " / " for Welsh authorities
        df_filtered['Clean_Area_Name'] = df_filtered['Area_Name'].str.split(' / ').str[0]
        
        # Convert Change_Percent to numeric, handling any non-numeric values
        df_filtered['Change_Percent'] = pd.to_numeric(df_filtered['Change_Percent'], errors='coerce')
        
        # Remove any rows where Change_Percent couldn't be converted
        df_filtered = df_filtered.dropna(subset=['Change_Percent'])
        
        print(f"Filtered to {len(df_filtered)} Local Authority areas")
        print(f"Sample areas: {df_filtered['Clean_Area_Name'].head().tolist()}")
        print(f"Change range: {df_filtered['Change_Percent'].min():.1f}% to {df_filtered['Change_Percent'].max():.1f}%")
        
        return df_filtered
    
    except Exception as e:
        print(f"Error loading Excel file: {e}")
        print("Please check the file path and make sure it has at least 6 sheets")
        return None

def get_uk_boundaries():
    """Download UK Local Authority boundaries from ONS"""
    
    boundaries_path = "uk_local_authorities.geojson"
    
    if os.path.exists(boundaries_path):
        print("Loading existing boundary data...")
        gdf = gpd.read_file(boundaries_path)
    else:
        print("Downloading UK Local Authority boundaries...")
        
        # Try multiple sources for boundary data
        urls_to_try = [
            # ONS 2023 boundaries
            "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_December_2023_Boundaries_UK_BFC/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson",
            # Backup: 2021 boundaries
            "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_December_2021_UK_BFC_2022/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson",
            # Fallback: GitHub source
            "https://raw.githubusercontent.com/martinjc/UK-GeoJSON/master/json/administrative/gb/lad.json"
        ]
        
        gdf = None
        for url in urls_to_try:
            try:
                print(f"Trying: {url[:60]}...")
                gdf = gpd.read_file(url)
                print(f"Success! Downloaded {len(gdf)} boundaries")
                
                # Save for future use
                gdf.to_file(boundaries_path, driver='GeoJSON')
                break
                
            except Exception as e:
                print(f"Failed: {str(e)[:100]}...")
                continue
        
        if gdf is None:
            print("Could not download boundary data from any source.")
            return None
    
    # Standardize column names - check what columns we have
    print(f"Boundary data columns: {gdf.columns.tolist()}")
    
    # Try to find the name column - different sources use different names
    name_columns = ['LAD23NM', 'LAD21NM', 'LAD20NM', 'LAD13NM', 'LAD_NAME', 'name', 'NAME']
    code_columns = ['LAD23CD', 'LAD21CD', 'LAD20CD', 'LAD13CD', 'LAD_CODE', 'code', 'CODE']
    
    name_col = None
    code_col = None
    
    for col in name_columns:
        if col in gdf.columns:
            name_col = col
            break
    
    for col in code_columns:
        if col in gdf.columns:
            code_col = col
            break
    
    if name_col is None:
        print("Warning: Could not find name column in boundary data")
        print("Available columns:", gdf.columns.tolist())
        return None
        
    if code_col is None:
        print("Warning: Could not find code column in boundary data")
        print("Available columns:", gdf.columns.tolist())
        return None
    
    # Rename columns to standard names
    gdf = gdf.rename(columns={name_col: 'LAD_NAME', code_col: 'LAD_CODE'})
    
    print(f"Using '{name_col}' as name column and '{code_col}' as code column")
    
    return gdf

def merge_data_with_boundaries(voa_df, boundaries_gdf):
    """Merge VOA data with geographic boundaries"""
    
    print(f"Merging {len(voa_df)} VOA records with {len(boundaries_gdf)} boundaries...")
    
    # Clean up authority names for better matching
    voa_df['clean_name'] = voa_df['Clean_Area_Name'].str.strip().str.upper()
    boundaries_gdf['clean_name'] = boundaries_gdf['LAD_NAME'].str.strip().str.upper()
    
    # Special handling for common name variations
    name_replacements = {
        'COUNTY DURHAM UA': 'COUNTY DURHAM',
        'CITY OF LONDON': 'CITY OF LONDON',
        'KINGSTON UPON HULL, CITY OF UA': 'KINGSTON UPON HULL',
        'BRISTOL, CITY OF UA': 'BRISTOL',
        'HEREFORDSHIRE, COUNTY OF UA': 'HEREFORDSHIRE',
        'ISLE OF ANGLESEY': 'ANGLESEY',
        'RHONDDA CYNON TAF': 'RHONDDA CYNON TAFF',
        'VALE OF GLAMORGAN': 'THE VALE OF GLAMORGAN',
        # Add more as needed based on your data
    }
    
    for old_name, new_name in name_replacements.items():
        voa_df.loc[voa_df['clean_name'] == old_name, 'clean_name'] = new_name
        boundaries_gdf.loc[boundaries_gdf['clean_name'] == old_name, 'clean_name'] = new_name
    
    # Merge the datasets
    merged = boundaries_gdf.merge(voa_df, on='clean_name', how='left')
    
    successful_matches = merged.dropna(subset=['Change_Percent']).shape[0]
    print(f"Successfully merged: {successful_matches} authorities")
    print(f"No data for: {len(boundaries_gdf) - successful_matches} authorities")
    
    # Show unmatched VOA records
    unmatched_voa = voa_df[~voa_df['clean_name'].isin(boundaries_gdf['clean_name'])]
    if len(unmatched_voa) > 0:
        print(f"\nUnmatched VOA authorities ({len(unmatched_voa)}):")
        for _, row in unmatched_voa[['Clean_Area_Name', 'clean_name']].head(10).iterrows():
            print(f"  '{row['Clean_Area_Name']}'")
    
    # Show unmatched boundary records  
    unmatched_boundaries = boundaries_gdf[~boundaries_gdf['clean_name'].isin(voa_df['clean_name'])]
    if len(unmatched_boundaries) > 5:  # Only show if many unmatched
        print(f"\nSample unmatched boundaries ({len(unmatched_boundaries)} total):")
        for _, row in unmatched_boundaries[['LAD_NAME', 'clean_name']].head(5).iterrows():
            print(f"  '{row['LAD_NAME']}'")
    
    return merged

def create_choropleth_map(merged_gdf, title="Business Rates Revaluation Impact"):
    """Create interactive Folium choropleth map"""
    
    # Calculate center of UK for map positioning
    bounds = merged_gdf.bounds
    center_lat = (bounds.miny.min() + bounds.maxy.max()) / 2
    center_lon = (bounds.minx.min() + bounds.maxx.max()) / 2
    
    # Create base map
    m = folium.Map(
        location=[54.5, -2.5],  # Center of UK
        zoom_start=6,
        tiles='CartoDB positron'
    )
    
    # Calculate color scale
    values = merged_gdf['Change_Percent'].dropna()
    if len(values) == 0:
        print("No valid data found for mapping!")
        return None
    
    vmin, vmax = values.min(), values.max()
    print(f"Change range: {vmin:.1f}% to {vmax:.1f}%")
    
    # Create choropleth
    choropleth = folium.Choropleth(
        geo_data=merged_gdf,
        name='Business Rates Change',
        data=merged_gdf,
        columns=['LAD_CODE', 'Change_Percent'],  # Use code for joining
        key_on='feature.properties.LAD_CODE',
        fill_color='RdYlBu_r',  # Red-Yellow-Blue reversed (red = increases)
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name=f'{title} (%)',
        nan_fill_color='lightgray',
        nan_fill_opacity=0.3
    )
    
    choropleth.add_to(m)
    
    # Add detailed tooltips
    style_function = lambda x: {
        'fillColor': '#ffffff',
        'color': '#000000',
        'fillOpacity': 0,
        'weight': 0.1
    }
    
    highlight_function = lambda x: {
        'fillColor': '#000000',
        'color': '#000000',
        'fillOpacity': 0.50,
        'weight': 0.1
    }
    
    # Create custom tooltip content
    merged_gdf['tooltip_content'] = merged_gdf.apply(lambda row: 
        f"<b>{row['LAD_NAME']}</b><br>"
        f"Change: {row['Change_Percent']:.1f}%<br>"
        f"2023 Mean RV: £{row['RV_Mean_2023']:,.0f}k<br>"
        f"2026 Mean RV: £{row['RV_Mean_2026']:,.0f}k<br>"
        f"Properties: {row['Properties_2023']:,.0f}"
        if pd.notna(row['Change_Percent']) else f"<b>{row['LAD_NAME']}</b><br>No data available"
    , axis=1)
    
    tooltip = folium.GeoJsonTooltip(
        fields=['tooltip_content'],
        aliases=[''],
        localize=False,
        sticky=False,
        labels=False,
        style="""
            background-color: white;
            border: 2px solid black;
            border-radius: 3px;
            box-shadow: 3px;
        """,
        max_width=300,
    )
    
    folium.GeoJson(
        merged_gdf,
        style_function=style_function,
        control=False,
        highlight_function=highlight_function,
        tooltip=tooltip
    ).add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Add title
    title_html = f'''
                 <h3 align="center" style="font-size:20px"><b>{title}</b></h3>
                 <p align="center" style="font-size:14px">Business Rates Revaluation 2026: Average Change by Local Authority (%)</p>
                 '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

def main():
    """Main function to create the business rates map"""
    
    # Step 1: Load your VOA data from sheet 6
    print("=== Loading VOA Data from Sheet 6 ===")
    excel_path = input("Enter path to your VOA Excel file: ")
    
    voa_df = load_voa_data(excel_path)
    if voa_df is None:
        return
    
    # Step 2: Get UK boundaries
    print("\n=== Loading Geographic Boundaries ===")
    boundaries_gdf = get_uk_boundaries()
    if boundaries_gdf is None:
        return
    
    # Step 3: Merge data
    print("\n=== Merging Data ===")
    merged_gdf = merge_data_with_boundaries(voa_df, boundaries_gdf)
    
    # Step 4: Create map
    print("\n=== Creating Interactive Map ===")
    map_obj = create_choropleth_map(
        merged_gdf,
        title="Business Rates Revaluation 2026"
    )
    
    if map_obj:
        # Save map
        output_file = "business_rates_revaluation_map.html"
        map_obj.save(output_file)
        print(f"\n✅ Map saved as '{output_file}'")
        print("Open this HTML file in your browser to view the interactive map!")
        
        # Display summary statistics
        print(f"\n=== Summary Statistics ===")
        values = merged_gdf['Change_Percent'].dropna()
        print(f"Local Authorities with data: {len(values)}")
        print(f"Average change: {values.mean():.1f}%")
        print(f"Median change: {values.median():.1f}%")
        print(f"Range: {values.min():.1f}% to {values.max():.1f}%")
        
        # Show top/bottom authorities
        merged_with_data = merged_gdf.dropna(subset=['Change_Percent'])
        top_increases = merged_with_data.nlargest(5, 'Change_Percent')[['LAD_NAME', 'Change_Percent']].round(1)
        top_decreases = merged_with_data.nsmallest(5, 'Change_Percent')[['LAD_NAME', 'Change_Percent']].round(1)
        
        print(f"\nTop 5 increases:")
        for _, row in top_increases.iterrows():
            print(f"  {row['LAD_NAME']}: +{row['Change_Percent']:.1f}%")
        
        print(f"\nTop 5 decreases:")
        for _, row in top_decreases.iterrows():
            print(f"  {row['LAD_NAME']}: {row['Change_Percent']:.1f}%")
        
        # Additional insights
        increases = values[values > 0]
        decreases = values[values < 0]
        unchanged = values[values == 0]
        
        print(f"\n=== Distribution ===")
        print(f"Areas with increases: {len(increases)} ({len(increases)/len(values)*100:.1f}%)")
        print(f"Areas with decreases: {len(decreases)} ({len(decreases)/len(values)*100:.1f}%)")
        print(f"Areas unchanged: {len(unchanged)} ({len(unchanged)/len(values)*100:.1f}%)")

if __name__ == "__main__":
    main()

=== Loading VOA Data from Sheet 6 ===
Loaded data from sheet 6 with shape: (401, 14)
Filtered to 354 Local Authority areas
Sample areas: ['County Durham UA', 'Darlington UA', 'Hartlepool UA', 'Middlesbrough UA', 'Northumberland UA']
Change range: 4.6% to 101.9%

=== Loading Geographic Boundaries ===
Loading existing boundary data...
Boundary data columns: ['LAD13CD', 'LAD13CDO', 'LAD13NM', 'LAD13NMW', 'geometry']
Using 'LAD13NM' as name column and 'LAD13CD' as code column

=== Merging Data ===
Merging 354 VOA records with 380 boundaries...
Successfully merged: 286 authorities
No data for: 94 authorities

Unmatched VOA authorities (68):
  'Darlington UA'
  'Hartlepool UA'
  'Middlesbrough UA'
  'Northumberland UA'
  'Redcar and Cleveland UA'
  'Stockton-on-Tees UA'
  'Blackburn with Darwen UA'
  'Blackpool UA'
  'Cheshire East UA'
  'Cheshire West and Chester UA'

Sample unmatched boundaries (94 total):
  'Hartlepool'
  'Middlesbrough'
  'Redcar and Cleveland'
  'Stockton-on-Tees'
  'Da

In [10]:
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
import numpy as np
import requests
import zipfile
import os
from pathlib import Path

def load_voa_data(excel_path):
    """Load and clean VOA spreadsheet data from multiple sheets (6-10 for different sectors)"""
    try:
        sector_data = {}
        
        # Define sheet mappings
        sheet_mappings = {
            6: 'All Sectors',
            7: 'Retail',  # Assuming based on your description
            8: 'Office',
            9: 'Industrial',  # Common sector
            10: 'Other'  # Or whatever sector this represents
        }
        
        for sheet_num, sector_name in sheet_mappings.items():
            try:
                print(f"Loading {sector_name} data from sheet {sheet_num}...")
                
                # Read from the specified sheet (0-indexed, so subtract 1)
                df = pd.read_excel(excel_path, sheet_name=sheet_num-1, header=None)
                
                # Define column names based on your data structure
                column_names = [
                    'Area_Type', 'Col2', 'LA_Code_Old', 'ONS_Code', 'Area_Name',
                    'Properties_2023', 'RV_2023', 'RV_per_Property_2023', 'Multiplier_2023',
                    'Total_BR_2023', 'RV_per_Property_2026', 'Multiplier_2026', 
                    'Total_BR_2026', 'Change_Percent'
                ]
                
                df.columns = column_names
                
                # Filter for Local Authority areas only (LAUA)
                df_filtered = df[df['Area_Type'] == 'LAUA'].copy()
                
                # Clean area names - remove anything after " / " for Welsh authorities
                df_filtered['Clean_Area_Name'] = df_filtered['Area_Name'].str.split(' / ').str[0]
                
                # Convert Change_Percent to numeric, handling any non-numeric values
                df_filtered['Change_Percent'] = pd.to_numeric(df_filtered['Change_Percent'], errors='coerce')
                df_filtered['Properties_2023'] = pd.to_numeric(df_filtered['Properties_2023'], errors='coerce')
                df_filtered['RV_2023'] = pd.to_numeric(df_filtered['RV_2023'], errors='coerce')
                df_filtered['Total_BR_2026'] = pd.to_numeric(df_filtered['Total_BR_2026'], errors='coerce')
                
                # Remove any rows where Change_Percent couldn't be converted
                df_filtered = df_filtered.dropna(subset=['Change_Percent'])
                
                # Add sector identifier
                df_filtered['Sector'] = sector_name
                
                sector_data[sector_name] = df_filtered
                
                print(f"  Loaded {len(df_filtered)} authorities for {sector_name}")
                print(f"  Change range: {df_filtered['Change_Percent'].min():.1f}% to {df_filtered['Change_Percent'].max():.1f}%")
                
            except Exception as e:
                print(f"  Warning: Could not load sheet {sheet_num} ({sector_name}): {e}")
                continue
        
        if not sector_data:
            print("No sector data could be loaded!")
            return None
            
        print(f"\nSuccessfully loaded {len(sector_data)} sectors")
        return sector_data
    
    except Exception as e:
        print(f"Error loading Excel file: {e}")
        print("Please check the file path and make sure it has at least 10 sheets")
        return None

def get_uk_boundaries():
    """Download UK Local Authority boundaries from ONS"""
    
    boundaries_path = "uk_local_authorities.geojson"
    
    if os.path.exists(boundaries_path):
        print("Loading existing boundary data...")
        gdf = gpd.read_file(boundaries_path)
    else:
        print("Downloading UK Local Authority boundaries...")
        
        # Try multiple sources for boundary data
        urls_to_try = [
            # ONS 2023 boundaries
            "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_December_2023_Boundaries_UK_BFC/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson",
            # Backup: 2021 boundaries
            "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_December_2021_UK_BFC_2022/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson",
            # Fallback: GitHub source
            "https://raw.githubusercontent.com/martinjc/UK-GeoJSON/master/json/administrative/gb/lad.json"
        ]
        
        gdf = None
        for url in urls_to_try:
            try:
                print(f"Trying: {url[:60]}...")
                gdf = gpd.read_file(url)
                print(f"Success! Downloaded {len(gdf)} boundaries")
                
                # Save for future use
                gdf.to_file(boundaries_path, driver='GeoJSON')
                break
                
            except Exception as e:
                print(f"Failed: {str(e)[:100]}...")
                continue
        
        if gdf is None:
            print("Could not download boundary data from any source.")
            return None
    
    # Standardize column names - check what columns we have
    print(f"Boundary data columns: {gdf.columns.tolist()}")
    
    # Try to find the name column - different sources use different names
    name_columns = ['LAD23NM', 'LAD21NM', 'LAD20NM', 'LAD13NM', 'LAD_NAME', 'name', 'NAME']
    code_columns = ['LAD23CD', 'LAD21CD', 'LAD20CD', 'LAD13CD', 'LAD_CODE', 'code', 'CODE']
    
    name_col = None
    code_col = None
    
    for col in name_columns:
        if col in gdf.columns:
            name_col = col
            break
    
    for col in code_columns:
        if col in gdf.columns:
            code_col = col
            break
    
    if name_col is None:
        print("Warning: Could not find name column in boundary data")
        print("Available columns:", gdf.columns.tolist())
        return None
        
    if code_col is None:
        print("Warning: Could not find code column in boundary data")
        print("Available columns:", gdf.columns.tolist())
        return None
    
    # Rename columns to standard names
    gdf = gdf.rename(columns={name_col: 'LAD_NAME', code_col: 'LAD_CODE'})
    
    print(f"Using '{name_col}' as name column and '{code_col}' as code column")
    
    return gdf

def merge_data_with_boundaries(sector_data, boundaries_gdf):
    """Merge VOA sector data with geographic boundaries"""
    
    merged_sectors = {}
    
    for sector_name, voa_df in sector_data.items():
        print(f"\nMerging {sector_name}: {len(voa_df)} records with {len(boundaries_gdf)} boundaries...")
        
        # Clean up authority names for better matching
        voa_df = voa_df.copy()  # Avoid modifying original
        voa_df['clean_name'] = voa_df['Clean_Area_Name'].str.strip().str.upper()
        boundaries_temp = boundaries_gdf.copy()
        boundaries_temp['clean_name'] = boundaries_temp['LAD_NAME'].str.strip().str.upper()
        
        # Special handling for common name variations
        name_replacements = {
            'COUNTY DURHAM UA': 'COUNTY DURHAM',
            'CITY OF LONDON': 'CITY OF LONDON',
            'KINGSTON UPON HULL, CITY OF UA': 'KINGSTON UPON HULL',
            'BRISTOL, CITY OF UA': 'BRISTOL',
            'HEREFORDSHIRE, COUNTY OF UA': 'HEREFORDSHIRE',
            'ISLE OF ANGLESEY': 'ANGLESEY',
            'RHONDDA CYNON TAF': 'RHONDDA CYNON TAFF',
            'VALE OF GLAMORGAN': 'THE VALE OF GLAMORGAN',
        }
        
        for old_name, new_name in name_replacements.items():
            voa_df.loc[voa_df['clean_name'] == old_name, 'clean_name'] = new_name
            boundaries_temp.loc[boundaries_temp['clean_name'] == old_name, 'clean_name'] = new_name
        
        # Merge the datasets
        merged = boundaries_temp.merge(voa_df, on='clean_name', how='left')
        
        successful_matches = merged.dropna(subset=['Change_Percent']).shape[0]
        print(f"  Successfully merged: {successful_matches} authorities")
        
        merged_sectors[sector_name] = merged
    
    return merged_sectors

def create_multi_sector_map(merged_sectors, title="Business Rates Revaluation Impact by Sector"):
    """Create interactive Folium choropleth map with sector toggles"""
    
    # Create base map
    m = folium.Map(
        location=[54.5, -2.5],  # Center of UK
        zoom_start=6,
        tiles='CartoDB positron'
    )
    
    # Use the same color scheme for all sectors
    color_scheme = 'RdYlBu_r'  # Red-Yellow-Blue reversed (red = increases, blue = decreases)
    
    # Calculate overall min/max across all sectors for consistent scaling
    all_values = []
    for sector_name, merged_gdf in merged_sectors.items():
        values = merged_gdf['Change_Percent'].dropna()
        if len(values) > 0:
            all_values.extend(values.tolist())
    
    if not all_values:
        print("No valid data found for any sector!")
        return None
    
    overall_min, overall_max = min(all_values), max(all_values)
    print(f"Overall change range across all sectors: {overall_min:.1f}% to {overall_max:.1f}%")
    
    # Create layers for each sector
    first_sector = True
    
    for sector_name, merged_gdf in merged_sectors.items():
        
        # Calculate color scale for this sector
        values = merged_gdf['Change_Percent'].dropna()
        if len(values) == 0:
            print(f"No valid data found for {sector_name}!")
            continue
        
        vmin, vmax = values.min(), values.max()
        print(f"{sector_name} change range: {vmin:.1f}% to {vmax:.1f}%")
        
        # Create choropleth for this sector - add directly to map
        choropleth = folium.Choropleth(
            geo_data=merged_gdf,
            name=f'{sector_name}',
            data=merged_gdf,
            columns=['LAD_CODE', 'Change_Percent'],
            key_on='feature.properties.LAD_CODE',
            fill_color=color_scheme,
            fill_opacity=0.7,
            line_opacity=0.2,
            legend_name='Change (%)' if first_sector else None,  # Only show legend for first sector
            nan_fill_color='lightgray',
            nan_fill_opacity=0.3,
            bins=8,
        )
        
        # Add choropleth directly to map
        choropleth.add_to(m)
        
        # Hide all layers except the first one initially
        if not first_sector:
            # This will be handled by the layer control
            pass
        
        # Add tooltip layer for this sector
        def style_function(feature):
            return {
                'fillColor': 'transparent',
                'color': 'transparent',
                'weight': 0,
                'fillOpacity': 0
            }
        
        def highlight_function(feature):
            return {
                'fillColor': '#ffff00',
                'color': 'black',
                'weight': 2,
                'fillOpacity': 0.3
            }
        
        # Create tooltip layer
        tooltip = folium.GeoJsonTooltip(
            fields=['LAD_NAME', 'Change_Percent', 'Properties_2023', 'RV_2023', 'Total_BR_2026'],
            aliases=[
                'Authority:', 
                f'{sector_name} Change (%):', 
                'Properties:', 
                '2023 RV (£k):', 
                '2026 RV (£k):'
            ],
            localize=True,
            sticky=False,
            labels=True,
            style="""
                background-color: white;
                border: 2px solid black;
                border-radius: 3px;
                box-shadow: 3px;
                font-size: 12px;
                padding: 5px;
            """,
            max_width=300,
        )
        
        # Create tooltip layer with same name as choropleth so they toggle together
        tooltip_layer = folium.GeoJson(
            merged_gdf,
            name=f'{sector_name}_tooltips',
            style_function=style_function,
            highlight_function=highlight_function,
            tooltip=tooltip,
            show=False  # Hidden by default, controlled by layer control
        )
        
        tooltip_layer.add_to(m)
        
        first_sector = False
    
    # Add layer control for toggling between sectors
    folium.LayerControl(position='topright', collapsed=False).add_to(m)
    
    # Add title and instructions
    title_html = f'''
                 <h3 align="center" style="font-size:20px"><b>{title}</b></h3>
                 <p align="center" style="font-size:14px">Business Rates Revaluation 2026: Average Change by Local Authority (%)</p>
                 <p align="center" style="font-size:12px"><i>Use the layer control (top-right) to toggle between sectors</i></p>
                 '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    # Add single legend/info box
    legend_html = f'''
    <div style="position: fixed; 
                bottom: 50px; left: 50px; width: 220px; height: auto; 
                background-color: white; border:2px solid grey; z-index:9999; 
                font-size:12px; padding: 10px; border-radius: 5px;">
    <h4 style="margin-top:0;">Business Rates Change</h4>
    <p style="margin:3px 0;"><b style="color:#d73027;">Red areas:</b> Rate increases</p>
    <p style="margin:3px 0;"><b style="color:#4575b4;">Blue areas:</b> Rate decreases</p>
    <p style="margin:3px 0;"><b style="color:#ffffcc;">Yellow areas:</b> Minimal change</p>
    <p style="margin:8px 0 3px 0;"><b>Range:</b> {overall_min:.1f}% to {overall_max:.1f}%</p>
    <p style="margin:3px 0;"><i>Toggle sectors using layer control ↗</i></p>
    <p style="margin:3px 0;"><i>Hover over areas for details</i></p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    # Add JavaScript to sync choropleth and tooltip visibility
    sync_script = '''
    <script>
    // Wait for map to load
    setTimeout(function() {
        // Get all layer control checkboxes
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-selector');
        
        checkboxes.forEach(function(checkbox) {
            checkbox.addEventListener('change', function() {
                var layerName = this.nextSibling.textContent.trim();
                var tooltipLayerName = layerName + '_tooltips';
                
                // Find corresponding tooltip layer
                map.eachLayer(function(layer) {
                    if (layer.options && layer.options.layerName === tooltipLayerName) {
                        if (this.checked) {
                            map.addLayer(layer);
                        } else {
                            map.removeLayer(layer);
                        }
                    }
                }.bind(this));
            });
        });
    }, 1000);
    </script>
    '''
    m.get_root().html.add_child(folium.Element(sync_script))
    
    return m

def main():
    """Main function to create the business rates map with sector toggles"""
    
    # Step 1: Load your VOA data from multiple sheets
    print("=== Loading VOA Data from Multiple Sheets ===")
    excel_path = input("Enter path to your VOA Excel file: ")
    
    sector_data = load_voa_data(excel_path)
    if sector_data is None:
        return
    
    # Step 2: Get UK boundaries
    print("\n=== Loading Geographic Boundaries ===")
    boundaries_gdf = get_uk_boundaries()
    if boundaries_gdf is None:
        return
    
    # Step 3: Merge data for all sectors
    print("\n=== Merging Data for All Sectors ===")
    merged_sectors = merge_data_with_boundaries(sector_data, boundaries_gdf)
    
    # Step 4: Create interactive map with sector toggles
    print("\n=== Creating Multi-Sector Interactive Map ===")
    map_obj = create_multi_sector_map(
        merged_sectors,
        title="Business Rates Revaluation 2026"
    )
    
    if map_obj:
        # Save map
        output_file = "business_rates_sectors_map.html"
        map_obj.save(output_file)
        print(f"\n✅ Multi-sector map saved as '{output_file}'")
        print("Open this HTML file in your browser to view the interactive map!")
        print("Use the layer control in the top-right to toggle between sectors.")
        
        # Display summary statistics for all sectors
        print(f"\n=== Summary Statistics by Sector ===")
        
        for sector_name, merged_gdf in merged_sectors.items():
            values = merged_gdf['Change_Percent'].dropna()
            if len(values) == 0:
                continue
                
            print(f"\n{sector_name}:")
            print(f"  Authorities with data: {len(values)}")
            print(f"  Average change: {values.mean():.1f}%")
            print(f"  Median change: {values.median():.1f}%")
            print(f"  Range: {values.min():.1f}% to {values.max():.1f}%")
            
            # Show top increases for this sector
            merged_with_data = merged_gdf.dropna(subset=['Change_Percent'])
            if len(merged_with_data) > 0:
                top_increases = merged_with_data.nlargest(3, 'Change_Percent')[['LAD_NAME', 'Change_Percent']]
                print(f"  Top 3 increases:")
                for _, row in top_increases.iterrows():
                    print(f"    {row['LAD_NAME']}: +{row['Change_Percent']:.1f}%")
        
        # Cross-sector comparison
        print(f"\n=== Cross-Sector Insights ===")
        sector_stats = {}
        for sector_name, merged_gdf in merged_sectors.items():
            values = merged_gdf['Change_Percent'].dropna()
            if len(values) > 0:
                sector_stats[sector_name] = {
                    'mean': values.mean(),
                    'count': len(values)
                }
        
        if sector_stats:
            print("Average change by sector:")
            for sector, stats in sorted(sector_stats.items(), key=lambda x: x[1]['mean'], reverse=True):
                print(f"  {sector}: {stats['mean']:.1f}% (n={stats['count']})")

if __name__ == "__main__":
    main()

=== Loading VOA Data from Multiple Sheets ===
Loading All Sectors data from sheet 6...
  Loaded 354 authorities for All Sectors
  Change range: 4.6% to 101.9%
Loading Retail data from sheet 7...
  Loaded 354 authorities for Retail
  Change range: -26.5% to 39.6%
Loading Office data from sheet 8...
  Loaded 354 authorities for Office
  Change range: 7.3% to 42.5%
Loading Industrial data from sheet 9...
  Loaded 354 authorities for Industrial
  Change range: -9.4% to 42.4%
Loading Other data from sheet 10...
  Loaded 354 authorities for Other
  Change range: -7.3% to 229.5%

Successfully loaded 5 sectors

=== Loading Geographic Boundaries ===
Loading existing boundary data...
Boundary data columns: ['LAD13CD', 'LAD13CDO', 'LAD13NM', 'LAD13NMW', 'geometry']
Using 'LAD13NM' as name column and 'LAD13CD' as code column

=== Merging Data for All Sectors ===

Merging All Sectors: 354 records with 380 boundaries...
  Successfully merged: 286 authorities

Merging Retail: 354 records with 380 bou

In [11]:
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
import numpy as np
import requests
import zipfile
import os
from pathlib import Path
import json

def load_voa_data(excel_path):
    """Load and clean VOA spreadsheet data from multiple sheets (6-10 for different sectors)"""
    try:
        sector_data = {}
        
        # Define sheet mappings
        sheet_mappings = {
            6: 'All Sectors',
            7: 'Retail',  
            8: 'Office',
            9: 'Industrial',  
            10: 'Other'  
        }
        
        for sheet_num, sector_name in sheet_mappings.items():
            try:
                print(f"Loading {sector_name} data from sheet {sheet_num}...")
                
                # Read from the specified sheet (0-indexed, so subtract 1)
                df = pd.read_excel(excel_path, sheet_name=sheet_num-1, header=None)
                
                # Define column names based on your data structure
                column_names = [
                    'Area_Type', 'Col2', 'LA_Code_Old', 'ONS_Code', 'Area_Name',
                    'Properties_2023', 'RV_2023', 'RV_per_Property_2023', 'Multiplier_2023',
                    'Total_BR_2023', 'RV_per_Property_2026', 'Multiplier_2026', 
                    'Total_BR_2026', 'Change_Percent'
                ]
                
                df.columns = column_names
                
                # Filter for Local Authority areas only (LAUA)
                df_filtered = df[df['Area_Type'] == 'LAUA'].copy()
                
                # Clean area names - remove anything after " / " for Welsh authorities
                df_filtered['Clean_Area_Name'] = df_filtered['Area_Name'].str.split(' / ').str[0]
                
                # Convert Change_Percent to numeric, handling any non-numeric values
                df_filtered['Change_Percent'] = pd.to_numeric(df_filtered['Change_Percent'], errors='coerce')
                df_filtered['Properties_2023'] = pd.to_numeric(df_filtered['Properties_2023'], errors='coerce')
                df_filtered['RV_2023'] = pd.to_numeric(df_filtered['RV_2023'], errors='coerce')
                df_filtered['Total_BR_2026'] = pd.to_numeric(df_filtered['Total_BR_2026'], errors='coerce')
                
                # Remove any rows where Change_Percent couldn't be converted
                df_filtered = df_filtered.dropna(subset=['Change_Percent'])
                
                # Add sector identifier
                df_filtered['Sector'] = sector_name
                
                sector_data[sector_name] = df_filtered
                
                print(f"  Loaded {len(df_filtered)} authorities for {sector_name}")
                print(f"  Change range: {df_filtered['Change_Percent'].min():.1f}% to {df_filtered['Change_Percent'].max():.1f}%")
                
            except Exception as e:
                print(f"  Warning: Could not load sheet {sheet_num} ({sector_name}): {e}")
                continue
        
        if not sector_data:
            print("No sector data could be loaded!")
            return None
            
        print(f"\nSuccessfully loaded {len(sector_data)} sectors")
        return sector_data
    
    except Exception as e:
        print(f"Error loading Excel file: {e}")
        print("Please check the file path and make sure it has at least 10 sheets")
        return None

def get_uk_boundaries():
    """Download UK Local Authority boundaries from ONS"""
    
    boundaries_path = "uk_local_authorities.geojson"
    
    if os.path.exists(boundaries_path):
        print("Loading existing boundary data...")
        gdf = gpd.read_file(boundaries_path)
    else:
        print("Downloading UK Local Authority boundaries...")
        
        # Try multiple sources for boundary data
        urls_to_try = [
            # ONS 2023 boundaries
            "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_December_2023_Boundaries_UK_BFC/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson",
            # Backup: 2021 boundaries
            "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/Local_Authority_Districts_December_2021_UK_BFC_2022/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson",
            # Fallback: GitHub source
            "https://raw.githubusercontent.com/martinjc/UK-GeoJSON/master/json/administrative/gb/lad.json"
        ]
        
        gdf = None
        for url in urls_to_try:
            try:
                print(f"Trying: {url[:60]}...")
                gdf = gpd.read_file(url)
                print(f"Success! Downloaded {len(gdf)} boundaries")
                
                # Save for future use
                gdf.to_file(boundaries_path, driver='GeoJSON')
                break
                
            except Exception as e:
                print(f"Failed: {str(e)[:100]}...")
                continue
        
        if gdf is None:
            print("Could not download boundary data from any source.")
            return None
    
    # Standardize column names - check what columns we have
    print(f"Boundary data columns: {gdf.columns.tolist()}")
    
    # Try to find the name column - different sources use different names
    name_columns = ['LAD23NM', 'LAD21NM', 'LAD20NM', 'LAD13NM', 'LAD_NAME', 'name', 'NAME']
    code_columns = ['LAD23CD', 'LAD21CD', 'LAD20CD', 'LAD13CD', 'LAD_CODE', 'code', 'CODE']
    
    name_col = None
    code_col = None
    
    for col in name_columns:
        if col in gdf.columns:
            name_col = col
            break
    
    for col in code_columns:
        if col in gdf.columns:
            code_col = col
            break
    
    if name_col is None:
        print("Warning: Could not find name column in boundary data")
        print("Available columns:", gdf.columns.tolist())
        return None
        
    if code_col is None:
        print("Warning: Could not find code column in boundary data")
        print("Available columns:", gdf.columns.tolist())
        return None
    
    # Rename columns to standard names
    gdf = gdf.rename(columns={name_col: 'LAD_NAME', code_col: 'LAD_CODE'})
    
    print(f"Using '{name_col}' as name column and '{code_col}' as code column")
    
    return gdf

def merge_data_with_boundaries(sector_data, boundaries_gdf):
    """Merge VOA sector data with geographic boundaries"""
    
    merged_sectors = {}
    
    for sector_name, voa_df in sector_data.items():
        print(f"\nMerging {sector_name}: {len(voa_df)} records with {len(boundaries_gdf)} boundaries...")
        
        # Clean up authority names for better matching
        voa_df = voa_df.copy()  # Avoid modifying original
        voa_df['clean_name'] = voa_df['Clean_Area_Name'].str.strip().str.upper()
        boundaries_temp = boundaries_gdf.copy()
        boundaries_temp['clean_name'] = boundaries_temp['LAD_NAME'].str.strip().str.upper()
        
        # Special handling for common name variations
        name_replacements = {
            'COUNTY DURHAM UA': 'COUNTY DURHAM',
            'CITY OF LONDON': 'CITY OF LONDON',
            'KINGSTON UPON HULL, CITY OF UA': 'KINGSTON UPON HULL',
            'BRISTOL, CITY OF UA': 'BRISTOL',
            'HEREFORDSHIRE, COUNTY OF UA': 'HEREFORDSHIRE',
            'ISLE OF ANGLESEY': 'ANGLESEY',
            'RHONDDA CYNON TAF': 'RHONDDA CYNON TAFF',
            'VALE OF GLAMORGAN': 'THE VALE OF GLAMORGAN',
        }
        
        for old_name, new_name in name_replacements.items():
            voa_df.loc[voa_df['clean_name'] == old_name, 'clean_name'] = new_name
            boundaries_temp.loc[boundaries_temp['clean_name'] == old_name, 'clean_name'] = new_name
        
        # Merge the datasets
        merged = boundaries_temp.merge(voa_df, on='clean_name', how='left')
        
        successful_matches = merged.dropna(subset=['Change_Percent']).shape[0]
        print(f"  Successfully merged: {successful_matches} authorities")
        
        merged_sectors[sector_name] = merged
    
    return merged_sectors

def create_single_sector_map(merged_gdf, sector_name, title_prefix="Business Rates Revaluation"):
    """Create a single sector map"""
    
    # Create base map
    m = folium.Map(
        location=[54.5, -2.5],
        zoom_start=6,
        tiles='CartoDB positron'
    )
    
    # Calculate color scale
    values = merged_gdf['Change_Percent'].dropna()
    if len(values) == 0:
        print(f"No valid data found for {sector_name}!")
        return None
    
    vmin, vmax = values.min(), values.max()
    
    # Create choropleth
    choropleth = folium.Choropleth(
        geo_data=merged_gdf,
        data=merged_gdf,
        columns=['LAD_CODE', 'Change_Percent'],
        key_on='feature.properties.LAD_CODE',
        fill_color='RdYlBu_r',
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name=f'{sector_name} Change (%)',
        nan_fill_color='lightgray',
        nan_fill_opacity=0.3,
        bins=8,
    )
    choropleth.add_to(m)
    
    # Add tooltips
    tooltip = folium.GeoJsonTooltip(
        fields=['LAD_NAME', 'Change_Percent', 'Properties_2023', 'RV_2023', 'Total_BR_2026'],
        aliases=['Authority:', f'{sector_name} Change (%):', 'Properties:', '2023 RV (£k):', '2026 RV (£k):'],
        localize=True,
        sticky=False,
        labels=True,
        style="""
            background-color: white;
            border: 2px solid black;
            border-radius: 3px;
            box-shadow: 3px;
            font-size: 12px;
            padding: 5px;
        """,
        max_width=300,
    )
    
    tooltip_layer = folium.GeoJson(
        merged_gdf,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': 'transparent',
            'weight': 0,
            'fillOpacity': 0
        },
        highlight_function=lambda x: {
            'fillColor': '#ffff00',
            'color': 'black',
            'weight': 2,
            'fillOpacity': 0.3
        },
        tooltip=tooltip
    )
    tooltip_layer.add_to(m)
    
    # Add title
    title_html = f'''
        <h3 align="center" style="font-size:20px"><b>{title_prefix}: {sector_name}</b></h3>
        <p align="center" style="font-size:14px">Average Change by Local Authority (%)</p>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    return m

def main():
    """Main function to create sector-specific business rates maps"""
    
    # Step 1: Load your VOA data from multiple sheets
    print("=== Loading VOA Data from Multiple Sheets ===")
    excel_path = input("Enter path to your VOA Excel file: ")
    
    sector_data = load_voa_data(excel_path)
    if sector_data is None:
        return
    
    # Step 2: Get UK boundaries
    print("\n=== Loading Geographic Boundaries ===")
    boundaries_gdf = get_uk_boundaries()
    if boundaries_gdf is None:
        return
    
    # Step 3: Merge data for all sectors
    print("\n=== Merging Data for All Sectors ===")
    merged_sectors = merge_data_with_boundaries(sector_data, boundaries_gdf)
    
    # Step 4: Create individual maps for each sector
    print("\n=== Creating Individual Sector Maps ===")
    
    created_maps = []
    
    for sector_name, merged_gdf in merged_sectors.items():
        values = merged_gdf['Change_Percent'].dropna()
        if len(values) == 0:
            print(f"Skipping {sector_name} - no valid data")
            continue
        
        print(f"Creating map for {sector_name}...")
        
        sector_map = create_single_sector_map(merged_gdf, sector_name)
        if sector_map:
            # Create filename
            safe_name = sector_name.lower().replace(' ', '_').replace('/', '')
            filename = f"business_rates_{safe_name}_map.html"
            sector_map.save(filename)
            created_maps.append((sector_name, filename))
            print(f"  Saved: {filename}")
    
    # Create an index page that links to all maps
    if created_maps:
        print("\n=== Creating Index Page ===")
        
        index_html = '''
        <!DOCTYPE html>
        <html>
        <head>
            <title>Business Rates Revaluation 2026 - Sector Analysis</title>
            <style>
                body { font-family: Arial, sans-serif; max-width: 800px; margin: 0 auto; padding: 20px; }
                .sector-link { 
                    display: block; 
                    padding: 15px; 
                    margin: 10px 0; 
                    background: #f0f0f0; 
                    text-decoration: none; 
                    border-radius: 5px; 
                    border-left: 5px solid #3388ff;
                    transition: background 0.3s;
                }
                .sector-link:hover { background: #e0e0e0; }
                .sector-name { font-size: 18px; font-weight: bold; color: #333; }
                .sector-desc { font-size: 14px; color: #666; margin-top: 5px; }
                h1 { color: #333; text-align: center; }
                .intro { text-align: center; color: #666; margin-bottom: 30px; }
            </style>
        </head>
        <body>
            <h1>Business Rates Revaluation 2026</h1>
            <p class="intro">Interactive maps showing the impact of business rates revaluation by sector</p>
        '''
        
        # Add statistics for each sector
        print("\n=== Sector Statistics ===")
        for sector_name, merged_gdf in merged_sectors.items():
            values = merged_gdf['Change_Percent'].dropna()
            if len(values) == 0:
                continue
                
            print(f"\n{sector_name}:")
            print(f"  Average change: {values.mean():.1f}%")
            print(f"  Range: {values.min():.1f}% to {values.max():.1f}%")
            print(f"  Authorities: {len(values)}")
        
        for sector_name, filename in created_maps:
            # Get sector stats
            merged_gdf = merged_sectors[sector_name]
            values = merged_gdf['Change_Percent'].dropna()
            avg_change = values.mean()
            min_change, max_change = values.min(), values.max()
            
            index_html += f'''
            <a href="{filename}" class="sector-link">
                <div class="sector-name">{sector_name}</div>
                <div class="sector-desc">
                    Average change: {avg_change:.1f}% | Range: {min_change:.1f}% to {max_change:.1f}% | {len(values)} authorities
                </div>
            </a>
            '''
        
        index_html += '''
        </body>
        </html>
        '''
        
        with open('business_rates_index.html', 'w') as f:
            f.write(index_html)
        
        print(f"\n✅ Created {len(created_maps)} sector maps!")
        print("📊 Open 'business_rates_index.html' to see all sectors")
        print("\nIndividual maps created:")
        for sector_name, filename in created_maps:
            print(f"  - {sector_name}: {filename}")

if __name__ == "__main__":
    main()

=== Loading VOA Data from Multiple Sheets ===
Loading All Sectors data from sheet 6...
  Loaded 354 authorities for All Sectors
  Change range: 4.6% to 101.9%
Loading Retail data from sheet 7...
  Loaded 354 authorities for Retail
  Change range: -26.5% to 39.6%
Loading Office data from sheet 8...
  Loaded 354 authorities for Office
  Change range: 7.3% to 42.5%
Loading Industrial data from sheet 9...
  Loaded 354 authorities for Industrial
  Change range: -9.4% to 42.4%
Loading Other data from sheet 10...
  Loaded 354 authorities for Other
  Change range: -7.3% to 229.5%

Successfully loaded 5 sectors

=== Loading Geographic Boundaries ===
Loading existing boundary data...
Boundary data columns: ['LAD13CD', 'LAD13CDO', 'LAD13NM', 'LAD13NMW', 'geometry']
Using 'LAD13NM' as name column and 'LAD13CD' as code column

=== Merging Data for All Sectors ===

Merging All Sectors: 354 records with 380 boundaries...
  Successfully merged: 286 authorities

Merging Retail: 354 records with 380 bou